## 19. Сравнение продовой LightGBM (с RD) и LightGBM без RD

**Идея:**

- В проде у нас стоит LightGBM на фичах `seq_5_15_30_60` **с RD-блоком**:
  - базовые RD-фичи: `rd_mom_*`, `rd_acceleration`, `rd_zscore_30`, `rd_ema_20`, `abs_rd`, `rd_regime`, `rd_regime_transition`;
  - плюс классические OHLCV/time-признаки и их rolling-агрегаты.
- В 18-м ноутбуке мы посмотрели, как работают **те же модели и окна**, но полностью **без RD-фичей** (как будто есть только Bybit: OHLCV + time).
- Здесь мы хотим сделать максимально честное head-to-head сравнение **двух LightGBM-моделей**:
  - **Prod-like LightGBM:** все 22 базовых фичи + rolling 5/15/30/60 (как в 16-м ноутбуке);
  - **No-RD LightGBM:** та же архитектура и гиперпараметры, тот же split, те же `ret_next` и бэктест, **но без любых RD-фичей и их rolling-агрегатов**.

Цель — измерить вклад RD-данных заказчика при полностью одинаковых условиях обучения, валидации, теста и логики сделок.

In [1]:
import sys, os, numpy as np, pandas as pd, warnings
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == '05_experiments' else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

COMMISSION_RT = 0.001
THRESHOLD = 0.75  # band 25-75
TARGET_COL = 'target'

print('Root:', _root)

Root: c:\project\trading_bot_2Engine


### 1. Загрузка данных и базовых фичей (как в 13/16/18)

In [2]:
labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path    = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')

df_raw = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df_raw.columns]

valid = df_raw.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]
valid['y'] = (valid[TARGET_COL] == 1).astype(int)
valid['date'] = pd.to_datetime(valid['datetime'], utc=True).dt.date

sort_col = 'datetime' if 'datetime' in valid.columns else 'timestamp'
valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)
valid = valid.dropna(subset=['ret_next'])

dates = sorted(valid['date'].unique())
assert len(dates) >= 10, f'Нужно >= 10 дней, найдено {len(dates)}'

train_dates = set(dates[:8])
val_dates   = set([dates[8]])
test_dates  = set([dates[9]])

print('Dates: train=', f'{min(train_dates)}..{max(train_dates)}', '| val=', dates[8], '| test=', dates[9])
print('Rows total:', len(valid))
print('BASE_FEATURES (22):', BASE_FEATURES)

Dates: train= 2026-02-01..2026-02-08 | val= 2026-02-09 | test= 2026-02-10
Rows total: 186063
BASE_FEATURES (22): ['rd_mom_1', 'rd_mom_5', 'rd_mom_10', 'rd_acceleration', 'rd_zscore_30', 'rd_ema_20', 'abs_rd', 'ret_1', 'ret_5', 'rsi_14', 'macd_signal', 'macd_hist', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'rd_regime', 'rd_regime_transition']


### 2. Определяем RD-фичи и варианты feature-set’ов

- `FEATURES_WITH_RD`: полная продовая конфигурация (22 base + rolling 5/15/30/60 для 10 ключевых фичей).
- `FEATURES_NO_RD`: тот же подход, но без RD-фичей и их rolling-агрегатов (как будто есть только Bybit).

In [3]:
# RD-фичи: всё, что строится напрямую из rd_value или его режимов
def is_rd_feature(c: str) -> bool:
    if c.startswith('rd_'):
        return True
    if c == 'abs_rd':
        return True
    return False

BASE_NO_RD = [c for c in BASE_FEATURES if not is_rd_feature(c)]
print('BASE_NO_RD (без RD):', len(BASE_NO_RD), BASE_NO_RD)

# Ключевые фичи для rolling в проде (как в 16-м ноутбуке)
SEQ_WINDOWS = [5, 15, 30, 60]
KEY_FEATS = BASE_FEATURES[:10]  # rd_mom_1 .. rsi_14

# KEY_FEATS без RD (для варианта без RD)
KEY_FEATS_NO_RD = [c for c in KEY_FEATS if c in BASE_NO_RD]
print('KEY_FEATS:', KEY_FEATS)
print('KEY_FEATS_NO_RD:', KEY_FEATS_NO_RD)

BASE_NO_RD (без RD): 13 ['ret_1', 'ret_5', 'rsi_14', 'macd_signal', 'macd_hist', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
KEY_FEATS: ['rd_mom_1', 'rd_mom_5', 'rd_mom_10', 'rd_acceleration', 'rd_zscore_30', 'rd_ema_20', 'abs_rd', 'ret_1', 'ret_5', 'rsi_14']
KEY_FEATS_NO_RD: ['ret_1', 'ret_5', 'rsi_14']


### 3. Rolling-фичи 5/15/30/60 (с RD и без RD) на одном и том же `valid`

Rolling считаем один раз на всём `valid` по `session_key`, затем формируем два набора фичей:

- `FEATURES_WITH_RD` — все rolling по `KEY_FEATS` (10 фичей, как в проде);
- `FEATURES_NO_RD` — rolling только по `KEY_FEATS_NO_RD` (без RD-компоненты).

In [4]:
grp = valid.groupby('session_key', group_keys=False)

# Rolling для всех KEY_FEATS (вариант с RD)
for w in SEQ_WINDOWS:
    for c in KEY_FEATS:
        valid[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
        valid[f'{c}_roll{w}_std']  = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

SEQ_FEATURES_WITH_RD = [f'{c}_roll{w}_{s}' for w in SEQ_WINDOWS for c in KEY_FEATS for s in ('mean', 'std')]
FEATURES_WITH_RD = BASE_FEATURES + SEQ_FEATURES_WITH_RD

# Rolling для варианта без RD: используем только KEY_FEATS_NO_RD
SEQ_FEATURES_NO_RD = [f'{c}_roll{w}_{s}' for w in SEQ_WINDOWS for c in KEY_FEATS_NO_RD for s in ('mean', 'std')]
FEATURES_NO_RD = BASE_NO_RD + SEQ_FEATURES_NO_RD

train_df = valid[valid['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
val_df   = valid[valid['date'].isin(val_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
test_df  = valid[valid['date'].isin(test_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)

print('FEATURES_WITH_RD:', len(FEATURES_WITH_RD))
print('FEATURES_NO_RD  :', len(FEATURES_NO_RD))
print('Rows (train/val/test):', len(train_df), len(val_df), len(test_df))

FEATURES_WITH_RD: 102
FEATURES_NO_RD  : 37
Rows (train/val/test): 146693 24014 15356


### 4. Бэктест (signal_flip) — та же логика, что в 13/15/16/16-prod

In [5]:
def backtest_prod_hold(proba, ret, session_ids, threshold=THRESHOLD, commission_rt=COMMISSION_RT):
    """signal_flip: BUY >= thr, SELL <= 1-thr, HOLD держим позицию."""
    pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i], prev = 1.0, 1.0
        elif pred[i] == 0:
            pos[i], prev = -1.0, -1.0
        else:
            pos[i] = prev
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    if session_ids is not None:
        sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    is_flip = pos_changed & (pos != 0) & (pos_prev != 0) & (pos * pos_prev < 0)
    fee_total = np.where(pos_changed, np.where(is_flip, commission_rt, commission_rt / 2.0), 0.0).sum()
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    avg_trade = float((pnl_net * 100) / trades) if trades > 0 else np.nan
    return {'trades': trades, 'net_%': float(pnl_net * 100), 'avg_%_per_trade': avg_trade}

ret_val = val_df['ret_next'].values
sess_val = val_df['session_key'].values
ret_test = test_df['ret_next'].values
sess_test = test_df['session_key'].values

### 5. Обучение двух LightGBM-моделей и сравнение

- `lgb_with_rd` — продовый вариант (с RD);
- `lgb_no_rd` — тот же LightGBM, те же гиперпараметры, но без RD-фичей.

In [6]:
def fit_lgb(feat_cols, label):
    df_tr, df_va, df_te = train_df, val_df, test_df
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(df_tr[feat_cols].fillna(0))
    Xva = scaler.transform(df_va[feat_cols].fillna(0))
    Xte = scaler.transform(df_te[feat_cols].fillna(0))
    ytr, yva, yte = df_tr['y'].values, df_va['y'].values, df_te['y'].values

    model = lgb.LGBMClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1,
    )
    model.fit(Xtr, ytr)

    p_val = model.predict_proba(Xva)[:, 1]
    p_test = model.predict_proba(Xte)[:, 1]

    auc_val = roc_auc_score(yva, p_val)
    auc_test = roc_auc_score(yte, p_test)

    bt_val = backtest_prod_hold(p_val, ret_val, sess_val, threshold=THRESHOLD)
    bt_test = backtest_prod_hold(p_test, ret_test, sess_test, threshold=THRESHOLD)

    return {
        'label': label,
        'auc_val': auc_val,
        'auc_test': auc_test,
        'net_val_%': bt_val['net_%'],
        'trades_val': bt_val['trades'],
        'avg_val_%_per_trade': bt_val['avg_%_per_trade'],
        'net_test_%': bt_test['net_%'],
        'trades_test': bt_test['trades'],
        'avg_test_%_per_trade': bt_test['avg_%_per_trade'],
    }

res_with_rd = fit_lgb(FEATURES_WITH_RD, 'lgb_with_rd (prod-like)')
res_no_rd   = fit_lgb(FEATURES_NO_RD,   'lgb_no_rd (no RD)')

cmp_df = pd.DataFrame([res_with_rd, res_no_rd])
cmp_df = cmp_df[['label', 'auc_val', 'auc_test', 'net_val_%', 'trades_val', 'avg_val_%_per_trade', 'net_test_%', 'trades_test', 'avg_test_%_per_trade']]
cmp_df

,label,auc_val,auc_test,net_val_%,trades_val,avg_val_%_per_trade,net_test_%,trades_test,avg_test_%_per_trade
0,lgb_with_rd (prod-like),0.906935,0.808684,2563.596778,1563,1.640177,1162.206502,851,1.365695
1,lgb_no_rd (no RD),0.732231,0.720743,679.339605,585,1.161264,444.891126,300,1.482970


### 6. Интерпретация

По таблице `cmp_df`:

| Модель | AUC val | AUC test | net_val_% | trades_val | avg_val_%_per_trade | net_test_% | trades_test | avg_test_%_per_trade |
|--------|---------|----------|-----------|------------|----------------------|-----------|------------|----------------------|
| **lgb_with_rd (prod-like)** | ≈ **0.907** | ≈ **0.809** | ≈ **+2564%** | 1563 | ≈ **1.64%** | ≈ **+1162%** | 851 | ≈ **1.37%** |
| **lgb_no_rd (no RD)**       | ≈ 0.732    | ≈ 0.721    | ≈ +679%     | 585  | ≈ 1.16%                 | ≈ +445%   | 300        | ≈ 1.48%                 |

1. **Вклад RD во фронт AUC:**
   - На валидации AUC растёт с ~0.73 до ~0.91 — это огромный скачок качества классификации.
   - На тесте AUC растёт с ~0.72 до ~0.81 — модель с RD заметно лучше разделяет исходы сделок не только на обучении, но и на OOT-дне.

2. **Вклад RD в суммарный PnL и количество сделок:**
   - На val-дне net_% растёт примерно с +680% до +2560%, количество сделок — с ~585 до ~1560.
   - На test-дне net_% растёт с ~+445% до ~+1160%, сделки — с ~300 до ~850.
   - То есть RD-фичи позволяют находить **больше прибыльных возможностей** при том же пороге и логике сделок.

3. **Средняя прибыль на сделку:**
   - На val: avg/trade растёт с ~1.16% до ~1.64% — RD не только даёт больше сделок, но и чуть повышает их “качество”.
   - На test: avg/trade без RD (~1.48%) чуть выше, чем с RD (~1.37%), но при этом сделок без RD в 2.8 раза меньше (300 vs 851), а суммарный PnL значительно ниже.
   - Для реальной торговли важнее **баланс**: высокая avg/trade + достаточное количество сделок; с RD этот баланс явно лучше.

4. **Итог по влиянию RD на продовую LightGBM:**
   - RD-блок радикально усиливает модель как классификатор (AUC +0.08–0.18) и даёт **существенный прирост суммарной доходности** при разумном количестве сделок.
   - Модель без RD на тех же окнах и гиперпараметрах остаётся рабочей (AUC ~0.72, avg/trade ~1.5%), но по совокупности AUC + net_% + trades заметно слабее.
   - Значит, RD-данные заказчика **реально добавляют рыночную альфу** поверх чистых биржевых данных и оправдывают свою интеграцию в продовую LightGBM.